# 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import models, transforms
from medmnist import PathMNIST, BloodMNIST
import numpy as np
import random
import copy

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 2. Reproduzierbarkeit

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Gerät: {DEVICE}")

NUM_SAMPLES_PER_CLASS = 50
NUM_UNKNOWN_SAMPLES = 450
UNKNOWN_LABEL = 9 

# Epochen-Zahlen (Für das finale Training)
NUM_EPOCHS_PHASE1 = 50
NUM_EPOCHS_PHASE2 = 30

# 3. Datenvorbereitung & Transformationen

In [ ]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 1.1 Ziel-Datensatz (PathMNIST)
train_dataset = PathMNIST(split='train', transform=data_transform, download=True, size=224)
val_dataset = PathMNIST(split='val', transform=data_transform, download=True, size=224)
test_dataset = PathMNIST(split='test', transform=data_transform, download=True, size=224)

def get_balanced_subset(dataset, n_per_class):
    labels = dataset.labels.flatten()
    indices = []
    for i in np.unique(labels):
        class_indices = np.where(labels == i)[0]
        selected_indices = np.random.choice(class_indices, n_per_class, replace=False)
        indices.extend(selected_indices)
    return Subset(dataset, indices)

small_train_ds = get_balanced_subset(train_dataset, NUM_SAMPLES_PER_CLASS)

# 1.2 "Unbekannt"-Datensatz (BloodMNIST)
blood_train_ds = BloodMNIST(split='train', transform=data_transform, download=True, size=224)
blood_indices = np.random.choice(len(blood_train_ds), NUM_UNKNOWN_SAMPLES, replace=False)
small_blood_ds = Subset(blood_train_ds, blood_indices)

class LabelWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset, new_label):
        self.dataset = dataset
        self.new_label = new_label
        
    def __getitem__(self, index):
        image, _ = self.dataset[index]
        return image, np.array([self.new_label], dtype=np.int64)
        
    def __len__(self):
        return len(self.dataset)

unknown_ds = LabelWrapper(small_blood_ds, UNKNOWN_LABEL)

# 4. Cross Validierung

In [ ]:
print("\n--- STARTE PHASE A: OPTUNA TUNING (STRATEGIE) ---")

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    
    target_labels = [train_dataset.labels[i][0] for i in small_train_ds.indices]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    fold_accuracies = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(target_labels)), target_labels)):
        # Daten für Phase 2 (Nur Zielklassen)
        fold_train_target = Subset(small_train_ds, train_idx)
        fold_val_target = Subset(small_train_ds, val_idx)
        
        # Daten für Phase 1 (Zielklassen + Unbekannt)
        fold_combined_train = ConcatDataset([fold_train_target, unknown_ds])
        
        loader_train_combined = DataLoader(fold_combined_train, batch_size=batch_size, shuffle=True)
        loader_train_target = DataLoader(fold_train_target, batch_size=batch_size, shuffle=True)
        loader_val = DataLoader(fold_val_target, batch_size=batch_size, shuffle=False)

        # --- Phase 1: Training auf 10 Klassen ---
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, 10)
        model = model.to(DEVICE)
        
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        best_p1_acc = 0.0
        best_p1_weights = None
        
        # Für Optuna reichen kurze Epochen (Zeit sparen)
        for epoch in range(15): 
            model.train()
            for inputs, targets in loader_train_combined:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()
                loss = criterion(model(inputs), targets)
                loss.backward()
                optimizer.step()
                
            # Validierung Phase 1 (Nur die ersten 9 Ausgänge prüfen)
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs)[:, :9], 1) 
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
            
            acc = accuracy_score(all_targets, all_preds)
            if acc > best_p1_acc:
                best_p1_acc = acc
                best_p1_weights = copy.deepcopy(model.state_dict())
                
        # --- Phase 2: Fine-Tuning auf 9 Klassen ---
        if best_p1_weights is not None:
            model.load_state_dict(best_p1_weights)
            
        model.fc = nn.Linear(model.fc.in_features, 9)
        model = model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        best_p2_acc = 0.0
        for epoch in range(15): # Kurzes Tuning
            model.train()
            for inputs, targets in loader_train_target:
                inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                optimizer.zero_grad()
                loss = criterion(model(inputs), targets)
                loss.backward()
                optimizer.step()
                
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, targets in loader_val:
                    inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
                    _, predicted = torch.max(model(inputs), 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
                    
            acc = accuracy_score(all_targets, all_preds)
            if acc > best_p2_acc:
                best_p2_acc = acc
                
        fold_accuracies.append(best_p2_acc)

    return np.mean(fold_accuracies)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10) # n_trials anpassen wie gewünscht

best_lr = study.best_params["lr"]
best_bs = study.best_params["batch_size"]

print(f"\nOptuna beendet. Beste Parameter gefunden:")
print(f"  Learning Rate: {best_lr:.6f}")
print(f"  Batch Size:    {best_bs}")

# 5. Finales Training

In [ ]:
print("\n--- STARTE PHASE B: FINALES TRAINING ---")

# Loader für alle Trainingsdaten
combined_train_ds_final = ConcatDataset([small_train_ds, unknown_ds])
final_train_loader_combined = DataLoader(combined_train_ds_final, batch_size=best_bs, shuffle=True)
final_train_loader_target = DataLoader(small_train_ds, batch_size=best_bs, shuffle=True)
final_val_loader = DataLoader(val_dataset, batch_size=best_bs, shuffle=False)

# --- FINALES TRAINING PHASE 1 ---
print(f"Trainiere Phase 1 (10 Klassen) auf vollen {len(combined_train_ds_final)} Samples...")
final_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
final_model.fc = nn.Linear(final_model.fc.in_features, 10)
final_model = final_model.to(DEVICE)

optimizer_final = optim.Adam(final_model.parameters(), lr=best_lr)
criterion_final = nn.CrossEntropyLoss()

best_final_p1_acc = 0.0
best_final_p1_weights = None

for epoch in range(NUM_EPOCHS_PHASE1):
    final_model.train()
    for inputs, targets in final_train_loader_combined:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_final.zero_grad()
        loss = criterion_final(final_model(inputs), targets)
        loss.backward()
        optimizer_final.step()
        
    final_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(final_model(inputs)[:, :9], 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    if acc > best_final_p1_acc:
        best_final_p1_acc = acc
        best_final_p1_weights = copy.deepcopy(final_model.state_dict())

# --- FINALES TRAINING PHASE 2 ---
print(f"\nTrainiere Phase 2 (9 Klassen) auf {len(small_train_ds)} Ziel-Samples...")
if best_final_p1_weights is not None:
    final_model.load_state_dict(best_final_p1_weights)

final_model.fc = nn.Linear(final_model.fc.in_features, 9)
final_model = final_model.to(DEVICE)
optimizer_final = optim.Adam(final_model.parameters(), lr=best_lr)

best_final_p2_acc = 0.0
best_final_p2_weights = None

for epoch in range(NUM_EPOCHS_PHASE2):
    final_model.train()
    for inputs, targets in final_train_loader_target:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        optimizer_final.zero_grad()
        loss = criterion_final(final_model(inputs), targets)
        loss.backward()
        optimizer_final.step()
        
    final_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in final_val_loader:
            inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
            _, predicted = torch.max(final_model(inputs), 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    if acc > best_final_p2_acc:
        best_final_p2_acc = acc
        best_final_p2_weights = copy.deepcopy(final_model.state_dict())
        print(f"Epoch {epoch+1}/{NUM_EPOCHS_PHASE2}: Neues bestes Modell! (Val Acc: {acc*100:.2f}%)")

final_model.load_state_dict(best_final_p2_weights)
print("Finales Training abgeschlossen. Bestes Modell geladen.")

# 6. Finale Evaluierung auf dem Test-Set

In [ ]:
print("\n--- STARTE PHASE C: FINALE AUSWERTUNG AUF TESTDATEN ---")
test_loader = DataLoader(test_dataset, batch_size=best_bs, shuffle=False)

final_model.eval()
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(DEVICE), targets.flatten().to(DEVICE).long()
        outputs = final_model(inputs)
        
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Metriken berechnen
test_acc = accuracy_score(all_targets, all_preds)
test_bal_acc = balanced_accuracy_score(all_targets, all_preds)
test_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')

print("="*50)
print("STRATEGIE ERGEBNISSE (TEST-SET)")
print("="*50)
print(f"Accuracy:      {test_acc*100:.2f}%")
print(f"Balanced Acc:  {test_bal_acc*100:.2f}%")
print(f"Precision:     {test_prec*100:.2f}%")
print(f"Recall:        {test_rec*100:.2f}%")
print(f"F1-Score:      {test_f1*100:.2f}%")
print(f"AUC-ROC:       {test_auc*100:.2f}%")
print("="*50)